# Forecast residual correction — frozen benchmark summary

This notebook is the **research scoreboard** against **benchmark v1** (`pkg.benchmark`).
It does **not** rebuild panels from SQL or `results/` CSVs. Headline metrics come from
`backtest` / `scoreboard` on the frozen matched universe.

**Locked Analysis B PRIMARY** (identical matched rows, n=1877, 5 origins):

| Model | Role | WMAPE |
|-------|------|------:|
| TS | Quantitative baseline | **43.88%** |
| Human | Sales Line Budget | **40.04%** |
| TS + XGB | Automated candidate | **37.23%** |
| Human + XGB | Human–machine candidate | **36.69%** |
| Integrated | Experimental | 40.14% |

Full narrative: [`docs/forecasting_findings.md`](../docs/forecasting_findings.md).
Rebuild freeze (rare): `python -m pkg.benchmark.freeze` · verify: `python -m pkg.benchmark.verify`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

_root = Path.cwd().resolve()
_src = _root.parent / "src" if _root.name == "notebooks" else _root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from pkg.benchmark import backtest, load_benchmark, scoreboard
from pkg.benchmark.config import EXPECTED_ANALYSIS_A_PRIMARY, EXPECTED_ANALYSIS_B_PRIMARY
from pkg.benchmark.dataset import prep_lags
from pkg.benchmark.evaluate import wmape
from pkg.benchmark.models import BUDGET_RESID_FEATURES, fit_xgb

ds = load_benchmark(verify_checksums=True)
print(
    f"benchmark v{ds.version}: ts={len(ds.ts_universe)} "
    f"budget={len(ds.budget_universe)} matched={len(ds.matched_universe)} "
    f"PRIMARY origins={ds.primary_origins}"
)


benchmark vv1: ts=11470 budget=4453 matched=3404 PRIMARY origins=[140404, 140407, 140410, 140501, 140504]


## 1. Matched scoreboard (Analysis B) — research baseline to beat

Identical `matched_universe` TEST rows (`same_horizon`).  
Train recipes: TS+XGB ← prior `ts_universe`; Human+XGB ← prior all `budget_universe`; Integrated ← prior matched only.


In [2]:
sb = scoreboard(
    ["ts", "human", "ts_xgb", "human_xgb", "integrated"],
    dataset=ds,
    universe="matched",
    eligibility="primary",
)
locked = EXPECTED_ANALYSIS_B_PRIMARY
sb["locked_wmape"] = sb["model"].map(
    {
        "ts": locked["ts"],
        "human": locked["human"],
        "ts_xgb": locked["ts_xgb"],
        "human_xgb": locked["human_xgb"],
        "integrated": locked["integrated"],
    }
)
sb["delta_vs_locked"] = sb["wmape"] - sb["locked_wmape"]
display(sb)
assert int(sb["n"].iloc[0]) == locked["n"]
assert (sb["delta_vs_locked"].abs() < 0.05).all(), "WMAPE drifted from locked baseline"
Markdown(
    f"**Best:** Human+XGB WMAPE **{sb.loc[sb.model=='human_xgb','wmape'].iloc[0]:.2f}** "
    f"vs Human **{sb.loc[sb.model=='human','wmape'].iloc[0]:.2f}** "
    f"vs TS **{sb.loc[sb.model=='ts','wmape'].iloc[0]:.2f}** (n={locked['n']})."
)


,model,rmse,mae,mape,wmape,bias,n,locked_wmape,delta_vs_locked
0,ts,24469.908821,7363.051678,58.737234,43.883479,1631.389451,1877,43.883479,4.851384e-07
1,human,21086.304463,6718.659256,57.498182,40.042928,1596.617129,1877,40.042928,-3.534944e-07
2,ts_xgb,19686.021156,6246.711771,58.816637,37.230140,835.275289,1877,37.230140,4.117013e-07
3,human_xgb,19286.411690,6156.880507,81.572099,36.694750,717.558867,1877,36.694750,-1.555333e-07
4,integrated,21517.854248,6735.484274,61.272193,40.143204,1651.516771,1877,40.143204,5.315593e-08


**Best:** Human+XGB WMAPE **36.69** vs Human **40.04** vs TS **43.88** (n=1877).

## 2. Stability by origin (matched Human vs Human+XGB)


In [3]:
r_h = backtest("human", dataset=ds)
r_hx = backtest("human_xgb", dataset=ds)
by_o = r_h.by_origin[["origin", "wmape", "n"]].rename(columns={"wmape": "Human_WMAPE"})
by_o = by_o.merge(
    r_hx.by_origin[["origin", "wmape"]].rename(columns={"wmape": "HumanML_WMAPE"}),
    on="origin",
)
by_o["rel_improvement_pct"] = (
    (by_o["Human_WMAPE"] - by_o["HumanML_WMAPE"]) / by_o["Human_WMAPE"] * 100
)
display(by_o)
print(
    f"origins improved: {(by_o['rel_improvement_pct'] > 0).sum()} / {len(by_o)}; "
    f"median rel={by_o['rel_improvement_pct'].median():.2f}% "
    f"(note: 140501 is a known failure)"
)


,origin,Human_WMAPE,n,HumanML_WMAPE,rel_improvement_pct
0,140404,36.853736,689,34.487884,6.419573
1,140407,39.730462,530,35.242947,11.294897
2,140410,47.491533,385,40.118173,15.525629
3,140501,34.682342,220,38.677828,-11.520233
4,140504,57.707834,53,49.609316,14.033653


origins improved: 4 / 5; median rel=11.29% (note: 140501 is a known failure)


## 3. All-Budget PRIMARY + negative controls (Analysis A)

Same eligibility; TEST = all Budget rows (not matched). Shows XGB lift is **not** explained by simple bias / AF / Ridge.


In [4]:
a_names = [
    "human",
    "bias_global",
    "bias_product",
    "bias_product_horizon",
    "af_ratio",
    "ridge",
    "human_xgb",
]
a_sb = scoreboard(a_names, dataset=ds, universe="budget", eligibility="primary")
human_w = float(a_sb.loc[a_sb.model == "human", "wmape"].iloc[0])
a_sb["vs_Human_pct"] = (human_w - a_sb["wmape"]) / human_w * 100
display(a_sb)
locked_a = EXPECTED_ANALYSIS_A_PRIMARY
assert int(a_sb["n"].iloc[0]) == locked_a["n"]


,model,rmse,mae,mape,wmape,bias,n,vs_Human_pct
0,human,20832.601959,6561.623205,71.595423,40.056187,1557.097426,1923,0.000000
1,bias_global,20973.650526,7079.738771,469.131695,43.219084,2758.656159,1923,-7.896149
2,bias_product,24418.602678,7498.470099,82.039902,45.775277,2795.009254,1923,-14.277670
3,bias_product_horizon,24315.897684,7567.912333,88.591829,46.199196,2468.982095,1923,-15.335979
4,af_ratio,22192.446700,7021.556733,77.255646,42.863905,2722.760424,1923,-7.009447
5,ridge,18222.855501,6750.627328,344.462001,41.209985,1205.356794,1923,-2.880448
6,human_xgb,19054.365434,6013.035944,87.623905,36.707273,702.006046,1923,8.360542


## 4. Train on all Budget history vs matched-only (Part 14)

Same matched TEST; Human+XGB trained on all prior Budget vs prior matched Budget only.


In [5]:
mu = prep_lags(ds.matched_universe)
bud = prep_lags(ds.budget_universe)
origins = ds.primary_origins
parts = []
for O in origins:
    test = mu.loc[mu["origin"].astype(int) == O].copy()
    if test.empty:
        continue
    train_all = bud.loc[bud["target_date"].astype(int) < O].copy()
    train_m = mu.loc[mu["target_date"].astype(int) < O].copy()
    if train_all.empty or train_m.empty:
        continue
    for label, tr in (("all_budget_universe", train_all), ("matched_only", train_m)):
        tr = tr.copy()
        tr["residual"] = tr["sales"] - tr["budget_forecast"]
        model = fit_xgb(BUDGET_RESID_FEATURES, tr)
        pred = np.maximum(
            0.0,
            test["budget_forecast"].to_numpy(dtype=float)
            + model.predict(test[BUDGET_RESID_FEATURES]),
        )
        parts.append(
            pd.DataFrame(
                {
                    "training_universe": label,
                    "actual": test["sales"].astype(float).to_numpy(),
                    "pred": pred,
                }
            )
        )

pp = pd.concat(parts, ignore_index=True)
part14 = pd.DataFrame(
    [
        {
            "training_universe": lab,
            "wmape": wmape(g["actual"], g["pred"]),
            "n": len(g),
        }
        for lab, g in pp.groupby("training_universe")
    ]
)
display(part14)


,training_universe,wmape,n
0,all_budget_universe,36.694750,1877
1,matched_only,38.600776,1877


## 5. Critical findings (frozen v1)

1. **Evaluation must be rolling-origin.** Date-split holdouts overstated TS+ML (~26% WMAPE lift); that result is discarded.
2. **On matched PRIMARY, Human+XGB is best** (36.69) among TS / Human / TS+XGB / Human+XGB / Integrated.
3. **Raw Human beats raw TS** on the same rows (~40.04 vs 43.88 WMAPE).
4. **TS+XGB helps vs TS** on this freeze (37.23) but still loses to Human+XGB.
5. **Integrated (matched-only train) is weaker** than Human+XGB trained on all Budget history (~40.14).
6. **Simple bias / AF / Ridge do not explain the XGB gain** (Analysis A: all worse WMAPE than Human except XGB).
7. **Lift is not uniform:** origin **140501** worsens; only ~half of SKUs improve on Analysis A.
8. **Working hypothesis:** moderate the **Human Budget** with ML; keep TS as comparator.

Custom candidates: `backtest(my_fn, origins=..., products=...)` with
`my_fn(train_df, test_df) -> forecasts` (default train = prior `budget_universe`).


In [6]:
# Example: evaluate a custom model against the locked Human+XGB baseline
def identity_human(train_df, test_df):
    """Negative control: return raw Budget (ignores train)."""
    return test_df["budget_forecast"].to_numpy(dtype=float)

custom = backtest(identity_human, dataset=ds)
baseline = backtest("human_xgb", dataset=ds)
print("custom (raw human) WMAPE", float(custom.overall["wmape"].iloc[0]))
print("locked Human+XGB WMAPE  ", float(baseline.overall["wmape"].iloc[0]))
print("to beat: below", EXPECTED_ANALYSIS_B_PRIMARY["human_xgb"])


custom (raw human) WMAPE 40.042927646505575
locked Human+XGB WMAPE   36.69474984446672
to beat: below 36.69475
